
# PySpark DataFrame Write Operations – Complete Guide

This notebook covers the most common ways to write a Spark DataFrame in real-world Data Engineering projects.

---

# Table of Contents

1. Basic Write
2. Write Modes
3. Write Different File Formats
4. Save Using `save()`
5. Save as Table
6. Insert Into Table
7. Partition By
8. Bucket By
9. Sort By
10. Repartition Before Writing
11. Coalesce Before Writing
12. Dynamic Partition Overwrite
13. Merge Schema
14. Overwrite Schema
15. Write with Compression
16. Write with Filters
17. Write to JDBC Database
18. Write Streaming Data
19. Best Practices
20. Interview Questions

---

# 1. Basic Write

```python
df.write.parquet("/mnt/data/employees")
```

---

# 2. Write Modes

Spark supports four write modes.

## Append

```python
df.write \
    .mode("append") \
    .parquet("/mnt/data/employees")
```

Adds new records.

---

## Overwrite

```python
df.write \
    .mode("overwrite") \
    .parquet("/mnt/data/employees")
```

Replaces existing data.

---

## Ignore

```python
df.write \
    .mode("ignore") \
    .parquet("/mnt/data/employees")
```

Does nothing if destination exists.

---

## Error (Default)

```python
df.write \
    .mode("error")
```

Throws an exception if destination exists.

---

# 3. Write Different File Formats

## Parquet

```python
df.write.parquet("/mnt/parquet")
```

---

## CSV

```python
df.write \
.option("header","true") \
.csv("/mnt/csv")
```

---

## JSON

```python
df.write.json("/mnt/json")
```

---

## ORC

```python
df.write.orc("/mnt/orc")
```

---

## Avro

```python
df.write \
.format("avro") \
.save("/mnt/avro")
```

---

## Delta

```python
df.write \
.format("delta") \
.save("/mnt/delta")
```

---

# 4. Using save()

```python
df.write \
.format("parquet") \
.save("/mnt/output")
```

Works with any supported format.

---

# 5. Save as Table

Creates a managed or external table.

```python
df.write \
.mode("overwrite") \
.saveAsTable("employee")
```

---

# 6. Insert Into Existing Table

Table must already exist.

```python
df.write \
.mode("append") \
.insertInto("employee")
```

Schema should match.

---

# 7. Partition By

Creates folder partitions.

```python
df.write \
.partitionBy("department") \
.parquet("/mnt/output")
```

Output

```
department=HR/
department=IT/
department=Finance/
```

## Multiple Partitions

```python
df.write \
.partitionBy("country","state")
```

Output

```
country=India/state=MH/
country=India/state=KA/
```

---

# 8. Bucket By

Used while saving tables.

```python
df.write \
.bucketBy(8,"department") \
.saveAsTable("employee_bucket")
```

Useful for joins.

---

# 9. Sort By

Sorts records inside each bucket.

```python
df.write \
.bucketBy(8,"department") \
.sortBy("salary") \
.saveAsTable("employee_bucket")
```

---

# 10. Repartition Before Writing

Increase or redistribute partitions.

```python
df.repartition(20) \
.write \
.parquet("/mnt/output")
```

Or

```python
df.repartition("department") \
.write \
.parquet("/mnt/output")
```

Use when

- Large datasets
- Better parallelism
- Even data distribution

---

# 11. Coalesce Before Writing

Reduce partitions.

```python
df.coalesce(1) \
.write \
.csv("/mnt/output")
```

Useful for

- Small datasets
- Single output file

Avoid for huge datasets.

---

# 12. Dynamic Partition Overwrite

Overwrite only affected partitions.

```python
spark.conf.set(
"spark.sql.sources.partitionOverwriteMode",
"dynamic"
)

df.write \
.mode("overwrite") \
.partitionBy("department") \
.parquet("/mnt/output")
```

Only matching partitions are overwritten.

---

# 13. Merge Schema

Automatically evolve schema.

```python
df.write \
.option("mergeSchema","true") \
.mode("append") \
.format("delta") \
.save("/mnt/delta")
```

Use when

- New columns arrive
- Schema evolves

---

# 14. Overwrite Schema

Replace table schema.

```python
df.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.save("/mnt/delta")
```

Use when

- Rename columns
- Remove columns
- Change datatype
- Complete redesign

---

# 15. Compression

## Snappy

```python
df.write \
.option("compression","snappy")
```

---

## Gzip

```python
df.write \
.option("compression","gzip")
```

---

## Bzip2

```python
df.write \
.option("compression","bzip2")
```

---

## Zstd

```python
df.write \
.option("compression","zstd")
```

---

# 16. Write Filtered Data

Write only required records.

```python
df.filter("salary > 50000") \
.write \
.parquet("/mnt/high_salary")
```

Or

```python
df.where("department='IT'") \
.write \
.parquet("/mnt/it")
```

Useful when

- Creating department-specific datasets
- Incremental loads
- Gold layer datasets

---

# 17. Write to JDBC Database

Example

```python
df.write \
.format("jdbc") \
.option("url","jdbc:mysql://host/db") \
.option("dbtable","employee") \
.option("user","user") \
.option("password","password") \
.mode("append") \
.save()
```

Works with

- SQL Server
- Oracle
- PostgreSQL
- MySQL

---

# 18. Write Streaming Data

```python
df.writeStream \
.format("delta") \
.outputMode("append") \
.option("checkpointLocation","/chk") \
.start("/mnt/output")
```

Used in Structured Streaming.

---

# Common Write Options

## Header

```python
.option("header","true")
```

---

## Delimiter

```python
.option("delimiter","|")
```

---

## Null Value

```python
.option("nullValue","NULL")
```

---

## Date Format

```python
.option("dateFormat","yyyy-MM-dd")
```

---

## Timestamp Format

```python
.option("timestampFormat","yyyy-MM-dd HH:mm:ss")
```

---

## Max Records Per File

```python
.option("maxRecordsPerFile",1000000)
```

Useful for avoiding very large files.

---

# Repartition vs Coalesce

| Repartition | Coalesce |
|-------------|----------|
| Shuffle | No shuffle (mostly) |
| Increase partitions | Only decrease partitions |
| Expensive | Faster |
| Better load balancing | Good for reducing files |

---

# mergeSchema vs overwriteSchema

| mergeSchema | overwriteSchema |
|--------------|----------------|
| Adds columns | Replaces schema |
| Schema evolution | Schema replacement |
| Append | Overwrite |
| Keeps old schema | Removes old schema |

---

# save() vs saveAsTable()

| save() | saveAsTable() |
|----------|---------------|
| Writes files | Creates Spark table |
| No metastore | Registered in catalog |
| Uses path | Uses table name |

---

# partitionBy vs repartition

| partitionBy | repartition |
|--------------|-------------|
| Storage layout | Execution layout |
| Creates folders | Changes Spark partitions |
| Happens during write | Happens before write |

---

# Best Practices

- Use **Parquet** or **Delta** for analytics.
- Partition only on **low-cardinality** columns (e.g., `country`, `year`, `month`).
- Avoid partitioning on **high-cardinality** columns like `customer_id`.
- Use **repartition()** before writing large datasets to improve parallelism.
- Use **coalesce()** only when reducing the number of output files.
- Prefer **Snappy** compression for Parquet and Delta.
- Use **Dynamic Partition Overwrite** for incremental loads.
- Use **mergeSchema** for schema evolution.
- Use **overwriteSchema** only for intentional schema redesigns.
- Avoid creating thousands of very small files.

---

# Common Interview Questions

### Q1. Difference between `save()` and `saveAsTable()`?

- `save()` writes files to a path.
- `saveAsTable()` creates a table in the metastore.

---

### Q2. Difference between `partitionBy()` and `repartition()`?

- `partitionBy()` organizes data on disk into folders.
- `repartition()` changes the number or distribution of Spark partitions in memory.

---

### Q3. When should you use `coalesce()`?

- Reduce partitions.
- Generate fewer output files.
- Suitable for small datasets or exports.

---

### Q4. When should you use `mergeSchema`?

- Adding new columns.
- Supporting schema evolution.

---

### Q5. When should you use `overwriteSchema`?

- Renaming or removing columns.
- Changing data types.
- Replacing the table schema.

---

# Summary

| Operation | Purpose |
|------------|---------|
| `save()` | Write files |
| `saveAsTable()` | Create Spark table |
| `insertInto()` | Insert into existing table |
| `partitionBy()` | Create partitioned folders |
| `bucketBy()` | Bucket data for joins |
| `sortBy()` | Sort records inside buckets |
| `repartition()` | Increase or redistribute partitions |
| `coalesce()` | Reduce partitions |
| `mergeSchema` | Schema evolution |
| `overwriteSchema` | Replace schema |
| `filter()` | Write subset of data |
| `format()` | Choose output format |
| `mode()` | Control write behavior |
| `writeStream()` | Stream data continuously |
| `jdbc` | Write to relational databases |